In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv(r"C:\Users\ryana\Downloads\DataSet\matches_full.csv")
pd.set_option('display.max_columns', None)

In [3]:
data.sample(5)

,Unnamed: 0,date,time,comp,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,opp formation,referee,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
1471,3,2022-09-03,16:15,La Liga,Matchweek 4,Sat,Away,L,1.0,2.0,Real Madrid,0.5,2.9,51.0,58579.0,Andrés Guardado,4-2-3-1,4-3-3,José Sánchez,Match Report,NaN,8.0,4.0,21.1,0.0,0,0,2023,Real Betis
3178,0,2020-09-12,21:00,La Liga,Matchweek 1,Sat,Away,W,2.0,0.0,Cádiz,0.8,0.7,48.0,NaN,Oier Sanjurjo,4-4-2,4-2-3-1,Isidro Díaz de Mera,Match Report,NaN,6.0,4.0,19.4,1.0,0,0,2021,Osasuna
3520,0,2020-09-12,16:00,La Liga,Matchweek 1,Sat,Home,D,0.0,0.0,Celta Vigo,0.4,0.6,50.0,NaN,Sergi Enrich,4-4-2,4-3-3,Valentín Pizarro,Match Report,NaN,8.0,1.0,21.5,1.0,0,0,2021,Eibar
1347,49,2023-04-29,18:30,La Liga,Matchweek 32,Sat,Home,W,4.0,2.0,Almería,2.8,1.3,68.0,58036.0,Karim Benzema,4-3-3,4-5-1,Guillermo Cuadra,Match Report,NaN,14.0,7.0,19.6,1.0,1,1,2023,Real Madrid
1483,22,2023-01-08,16:15,La Liga,Matchweek 16,Sun,Away,W,2.0,1.0,Rayo Vallecano,0.9,1.6,38.0,13119.0,Andrés Guardado,4-2-3-1,4-4-1-1,Isidro Díaz de Mera,Match Report,NaN,14.0,2.0,19.3,1.0,0,0,2023,Real Betis


### Data PreProcessing

#### Data Cleaning

In [4]:
data = data.drop(['Unnamed: 0'], axis=1)
data = data.drop(['comp'], axis=1)
data = data.drop(['captain'], axis=1)
data = data.drop(['referee'], axis=1)
data = data.drop(['match report'], axis=1)
data = data.drop(['notes'], axis=1)

In [5]:
# Standarized Value in 'formation' column
data['formation'].unique()

array(['4-2-3-1', '4-3-3', '4-1-4-1', '3-4-3', '5-3-2', '4-4-2', '5-4-1',
       '4-1-2-1-2◆', '4-4-1-1', '4-2-2-2', '4-5-1', '3-5-2', '3-5-1-1',
       '3-1-4-2', '3-4-3◆', '3-4-1-2', '4-1-3-2', '4-3-1-2', '3-2-4-1',
       '4-3-2-1', '4-2-4-0', '3-3-3-1'], dtype=object)

In [6]:
# Membersihkan simbol tidak diperlukan, misalnya ◆
data['formation'] = data['formation'].str.replace('◆', '', regex=False)

In [7]:
data['formation'] = data['formation'].replace('4-2-4-0', '4-2-4')

In [8]:
data['date'] = pd.to_datetime(data['date'])

In [9]:
data['time'] = pd.to_datetime(data['time'], format = '%H:%M').dt.time

In [10]:
data['opponent'] = data['opponent'].astype('category')

In [11]:
# change the data type to int
data['sh'] = data['sh'].astype(int)
data['sot'] = data['sot'].astype(int)
data['ga'] = data['ga'].astype(int)
data['gf'] = data['gf'].astype(int)
data['fk'] = data['fk'].astype(int)

In [12]:
data.sample(5)

,date,time,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,formation,opp formation,sh,sot,dist,fk,pk,pkatt,season,team
3540,2021-01-30,14:00:00,Matchweek 21,Sat,Home,L,0,2,Sevilla,0.9,1.5,49.0,NaN,4-4-2,4-3-3,17,6,19.2,0,0,0,2021,Eibar
200,2025-01-20,21:00:00,Matchweek 20,Mon,Away,L,0,4,Villarreal,0.5,2.4,57.0,14852.0,4-4-2,4-4-2,9,2,18.2,1,0,0,2025,Mallorca
2833,2021-05-12,22:00:00,Matchweek 36,Wed,Home,W,2,1,Real Sociedad,1.4,1.7,35.0,NaN,4-3-3,4-2-3-1,12,4,16.7,1,0,0,2021,Atletico Madrid
2541,2021-10-23,18:30:00,Matchweek 10,Sat,Away,D,2,2,Elche,0.8,0.9,43.0,13449.0,4-2-3-1,4-2-2-2,20,7,20.9,2,0,0,2022,Espanyol
3650,2019-12-14,21:00:00,Matchweek 17,Sat,Home,W,2,0,Osasuna,2.5,0.7,47.0,46033.0,4-4-2,4-5-1,19,9,15.7,0,0,0,2020,Atletico Madrid


#### Check NaN Value & Handling Missing Value

In [13]:
data.isnull().sum()

date               0
time               0
round              0
day                0
venue              0
result             0
gf                 0
ga                 0
opponent           0
xg                 0
xga                0
poss               0
attendance       974
formation          0
opp formation      0
sh                 0
sot                0
dist               3
fk                 0
pk                 0
pkatt              0
season             0
team               0
dtype: int64

In [14]:
data['attendance'] = data.groupby(['team', 'venue', 'opponent'])['attendance'].transform(
    lambda x: x.fillna(x.median())
)

C:\Users\ryana\AppData\Local\Temp\ipykernel_11540\321960465.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data['attendance'] = data.groupby(['team', 'venue', 'opponent'])['attendance'].transform(
C:\Users\ryana\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ryana\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ryana\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keep

In [15]:
# we still got the null value, then we will imputation with median techic
data['attendance'] = data['attendance'].fillna(data['attendance'].median())

In [16]:
# change the data type to int
data['attendance'] = data['attendance'].astype(int)

In [17]:
data['dist'] = data.groupby(['venue', 'team'])['dist'].transform(
    lambda x: x.fillna(x.mean())
)

In [18]:
data.isnull().sum()

date             0
time             0
round            0
day              0
venue            0
result           0
gf               0
ga               0
opponent         0
xg               0
xga              0
poss             0
attendance       0
formation        0
opp formation    0
sh               0
sot              0
dist             0
fk               0
pk               0
pkatt            0
season           0
team             0
dtype: int64

### Machine Learning

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
data_model = data.copy()

In [ ]:
data_model = data_model.drop('time', axis=1)

In [ ]:
data.sample()

In [21]:
df_encoded = data.copy()

In [22]:
categorical_cols = ['venue', 'day', 'formation', 'opp formation', 'opponent', 'team']

for col in categorical_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

In [23]:
result_map = {'W':2, 'D':1, 'L':0}
df_encoded['target_pred'] = df_encoded['result'].map(result_map)

In [24]:
# Hapus kolom yang tidak relevan
drop_cols = ['date', 'time', 'result', 'round', 'day', 'dist']  # karena redundant/target leak
df_encoded = df_encoded.drop(columns=drop_cols)

In [25]:
# Pisahkan fitur dan target
X = df_encoded.drop(columns=['target_pred'])
y = df_encoded['target_pred']

In [26]:
df_encoded.sample(3)

,venue,gf,ga,opponent,xg,xga,poss,attendance,formation,opp formation,sh,sot,fk,pk,pkatt,season,team,target_pred
553,1,5,0,0,2.5,1.3,71.0,68954,13,11,22,11,1,0,0,2024,21,2
4081,0,1,1,11,0.6,0.8,46.0,20185,17,17,8,2,1,0,0,2020,7,1
1685,1,1,2,3,0.7,1.9,51.0,13512,11,19,15,3,0,0,0,2023,19,0


In [27]:
# ----- 2. Feature Selection dengan Mutual Information -----
mi_scores = mutual_info_classif(X, y, discrete_features='auto')
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

print("\n🎯 Ranking Fitur berdasarkan Mutual Information:")
print(mi_series)


🎯 Ranking Fitur berdasarkan Mutual Information:
gf               0.287963
ga               0.285978
xg               0.095427
xga              0.093500
sot              0.069001
team             0.047292
pkatt            0.040473
opponent         0.031822
attendance       0.030741
venue            0.025557
opp formation    0.011095
sh               0.009113
pk               0.007255
formation        0.005242
season           0.000907
poss             0.000000
fk               0.000000
dtype: float64


In [31]:
# ----- 3. Feature Importance dari Gradient Boosting -----
model = GradientBoostingClassifier(max_depth=5, random_state=42)
model.fit(X, y)
gb_importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

print("\n🔥 Ranking Fitur berdasarkan Gradient Boosting Importance:")
print(gb_importance)


🔥 Ranking Fitur berdasarkan Gradient Boosting Importance:
ga               5.385367e-01
gf               4.602684e-01
xg               5.022126e-04
xga              4.182893e-04
season           1.264790e-04
attendance       3.703961e-05
team             3.259619e-05
opponent         2.712667e-05
formation        1.869710e-05
sot              1.624527e-05
sh               1.416453e-05
poss             6.963614e-07
pk               5.048194e-07
pkatt            4.486952e-07
fk               2.658806e-07
opp formation    1.587679e-07
venue            1.426469e-08
dtype: float64


#### Feature Engginering

In [ ]:
# result_map = {'W':2, 'D':1, 'L':0}
# data_model['target_pred'] = data_model['result'].map(result_map)

In [ ]:
# Poin dari result
result_map_point = {'W': 3, 'D': 1, 'L': 0}
data_model['points'] = data_model['result'].map(result_map_point)

In [ ]:
data_model.sample()

In [ ]:
#Checking Value == 0
def check_zero(data, kolom_list):
    value={}
    for kolom in kolom_list:
        jumlah_nol = (data[kolom] == 0).sum()
        value[kolom] = jumlah_nol
    return value

In [ ]:
critical_cols = ['poss', 'xga', 'xg', 'sh', 'pkatt']
hasil_cek = check_zero(data_model, critical_cols)

for kolom, jumlah in hasil_cek.items():
    print(f"Count Value '0' in column '{kolom}': {jumlah}")

In [ ]:
#create feature that indicate attacking power vs defensive power
data_model['xg_diff'] = data_model['xg'] - data_model['xga']

In [ ]:
# shooting Accuracy
data_model['sh_accuracy'] = data_model.apply(
    lambda row: row['sot'] / row['sh'] if row['sh'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['conversion_rate'] = data_model.apply(
    lambda row: row['gf'] / row['sh'] if row['sh'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['PCR'] = data_model.apply(
    lambda row: row['pk'] / row['pkatt'] if row['pkatt'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['finishing_efficiency'] = data_model.apply(
    lambda row: row['gf'] / row['xg'] if row['xg'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['sh_per_poss'] = data_model.apply(
    lambda row: row['sh'] / row['poss'] if row['poss'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['sot_per_xg'] = data_model.apply(
    lambda row: row['sot'] / row['xg'] if row['xg'] != 0 else 0,
    axis=1
)

In [ ]:
data_model['fk_per_sh'] = data_model.apply(
    lambda row: row['fk'] / row['sh'] if row['sh'] != 0 else 0,
    axis=1
)